In [ ]:
from tools import read_mask
from matplotlib import pyplot as plt
import cv2
import numpy as np
import json
import os
from PIL import Image
from monai.data import WSIReader
from cucim.clara import CuImage
import tqdm
import math
##################################CONFIGS##################################
svs_roots = '/mnt/c/Data/SVS file'
export_dir = '/mnt/f/cctgs_valid'
anno_prefix = 'annotations'
img_prefix = 'images'
reader = WSIReader(backend='cucim')

train_data_list = json.load(open('/mnt/d/Cpathdata/data/train_data.json', 'r'))
val_data_list = json.load(open('/mnt/d/Cpathdata/data/val_data.json', 'r'))
test_data_list = json.load(open('/mnt/d/Cpathdata/data/test_data.json', 'r'))
splits = ['train', 'validation']
os.makedirs(os.path.join(export_dir, anno_prefix), exist_ok=True)
os.makedirs(os.path.join(export_dir, img_prefix), exist_ok=True)

tile_size = 1024
threshold = 215
ratio = 0.8
stride = 512
###########################################################################



In [ ]:
train_set = set(train_data_list)
val_set = set(val_data_list)
test_set = set(test_data_list)

assert train_set & val_set == set()
assert train_set & test_set == set()
assert val_set & test_set == set()




In [ ]:

def extract_patch(reader:WSIReader, slide:CuImage, geodata, filename, split):
    level_count = reader.get_level_count(slide)

    if level_count == 4:
        lvl = 1
        ds = reader.get_downsample_ratio(wsi=slide, level=lvl)
    elif level_count == 3:
        lvl = 0
        ds = 2
    print(reader.get_mpp(slide, level=lvl), lvl)
    image, metadata = reader.get_data(wsi=slide, level=lvl)
    print(f'processing {filename}')
    print(f'metadata: {metadata}')

    if lvl == 0:
        image = cv2.resize(image.transpose(1, 2, 0), dsize=None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    H, W  = image.shape[-2:]
    print(image.shape)

    if H < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, H - tile_size + 1, stride))
        if (H - tile_size) % stride != 0:
            y_starts.append(H - tile_size)

    if W < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, W - tile_size + 1, stride))
        if (W - tile_size) % stride != 0:
            x_starts.append(W - tile_size)
    for y in y_starts:
        for x in x_starts:
            filename_ = filename.replace('.svs', ',') + f'x={math.floor(x * ds)},y={math.floor(y * ds)}.png'
            crop = image[..., y:y+tile_size, x:x+tile_size]
            
            bg = (crop[0, ...] >= threshold) & (crop[1, ...] >= threshold) & (crop[2, ...] >= threshold)
            fg = ~bg
            if np.mean(fg)<=ratio:
                continue

            mask = read_mask(geodata=geodata, location=[y*ds, x*ds], roi_size=(tile_size, tile_size), ds=ds)
            if np.sum(mask) == 0:
                continue
            # CHW
            crop = Image.fromarray(crop.transpose(1, 2, 0))

            crop.save(os.path.join(export_dir, img_prefix, split, filename_))
            mask.save(os.path.join(export_dir, anno_prefix, split, filename_))

In [ ]:
split = 'train'
os.makedirs(os.path.join(export_dir, anno_prefix, split), exist_ok=True)
os.makedirs(os.path.join(export_dir, img_prefix, split), exist_ok=True)
data = [os.path.join(svs_roots, item.replace('.png', '.svs')) for item in train_data_list]

for item in tqdm.tqdm(data):
    filename = os.path.basename(item)
    slide = reader.read(item)
    geodata = json.load(open(item.replace('.svs', '.geojson'), 'r'))

    extract_patch(reader=reader, slide=slide, geodata=geodata, filename=filename, split=split)


In [ ]:

stride = tile_size
def extract_patch(reader:WSIReader, slide:CuImage, geodata, filename, split):
    level_count = reader.get_level_count(slide)

    if level_count == 4:
        lvl = 1
        ds = reader.get_downsample_ratio(wsi=slide, level=lvl)
    elif level_count == 3:
        lvl = 0
        ds = 2
    print(reader.get_mpp(slide, level=lvl), lvl)
    image, metadata = reader.get_data(wsi=slide, level=lvl)
    print(f'processing {filename}')
    print(f'metadata: {metadata}')

    if lvl == 0:
        image = cv2.resize(image.transpose(1, 2, 0), dsize=None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    H, W  = image.shape[-2:]
    print(image.shape)

    if H < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, H - tile_size + 1, stride))
        if (H - tile_size) % stride != 0:
            y_starts.append(H - tile_size)

    if W < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, W - tile_size + 1, stride))
        if (W - tile_size) % stride != 0:
            x_starts.append(W - tile_size)
    for y in y_starts:
        for x in x_starts:
            filename_ = filename.replace('.svs', ',') + f'x={math.floor(x * ds)},y={math.floor(y * ds)}.png'
            crop = image[..., y:y+tile_size, x:x+tile_size]
            
            bg = (crop[0, ...] >= threshold) & (crop[1, ...] >= threshold) & (crop[2, ...] >= threshold)
            fg = ~bg
            if np.mean(fg)<=ratio:
                continue

            mask = read_mask(geodata=geodata, location=[y*ds, x*ds], roi_size=(tile_size, tile_size), ds=ds)
            if np.sum(mask) == 0:
                continue
            # CHW
            crop = Image.fromarray(crop.transpose(1, 2, 0))

            crop.save(os.path.join(export_dir, img_prefix, split, filename_))
            mask.save(os.path.join(export_dir, anno_prefix, split, filename_))

In [ ]:
os.makedirs(os.path.join(export_dir, anno_prefix, 'validation'), exist_ok=True)
os.makedirs(os.path.join(export_dir, img_prefix, 'validation'), exist_ok=True)
data = [os.path.join(svs_roots, item.replace('.png', '.svs')) for item in val_data_list]
stride = tile_size
for item in tqdm.tqdm(data):
    filename = os.path.basename(item)
    slide = reader.read(item)
    geodata = json.load(open(item.replace('.svs', '.geojson'), 'r'))

    extract_patch(reader=reader, slide=slide, geodata=geodata, filename=filename, split='validation')

In [ ]:

stride = tile_size
def extract_patch(reader:WSIReader, slide:CuImage, geodata, filename, split):
    level_count = reader.get_level_count(slide)

    if level_count == 4:
        lvl = 1
        ds = reader.get_downsample_ratio(wsi=slide, level=lvl)
    elif level_count == 3:
        lvl = 0
        ds = 2
    print(reader.get_mpp(slide, level=lvl), lvl)
    image, metadata = reader.get_data(wsi=slide, level=lvl)
    print(f'processing {filename}')
    print(f'metadata: {metadata}')

    if lvl == 0:
        image = cv2.resize(image.transpose(1, 2, 0), dsize=None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    H, W  = image.shape[-2:]
    print(image.shape)

    if H < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, H - tile_size + 1, stride))
        if (H - tile_size) % stride != 0:
            y_starts.append(H - tile_size)

    if W < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, W - tile_size + 1, stride))
        if (W - tile_size) % stride != 0:
            x_starts.append(W - tile_size)
    for y in y_starts:
        for x in x_starts:
            filename_ = filename.replace('.svs', ',') + f'x={math.floor(x * ds)},y={math.floor(y * ds)}.png'
            crop = image[..., y:y+tile_size, x:x+tile_size]
            
            bg = (crop[0, ...] >= threshold) & (crop[1, ...] >= threshold) & (crop[2, ...] >= threshold)
            fg = ~bg
            if np.mean(fg)<=ratio:
                continue

            mask = read_mask(geodata=geodata, location=[y*ds, x*ds], roi_size=(tile_size, tile_size), ds=ds)
            # if np.sum(mask) == 0:
            #     continue
            # CHW7
            crop = Image.fromarray(crop.transpose(1, 2, 0))

            crop.save(os.path.join(export_dir, img_prefix, split, filename_))
            mask.save(os.path.join(export_dir, anno_prefix, split, filename_))

In [ ]:

os.makedirs(os.path.join(export_dir, anno_prefix, 'test'), exist_ok=True)
os.makedirs(os.path.join(export_dir, img_prefix, 'test'), exist_ok=True)
data = [os.path.join(svs_roots, item.replace('.png', '.svs')) for item in test_data_list]
stride = tile_size
for item in tqdm.tqdm(data):
    filename = os.path.basename(item)
    slide = reader.read(item)
    geodata = json.load(open(item.replace('.svs', '.geojson'), 'r'))

    extract_patch(reader=reader, slide=slide, geodata=geodata, filename=filename, split='test')

In [ ]:

stride = tile_size
def extract_patch(reader:WSIReader, slide:CuImage, geodata, filename, split):
    level_count = reader.get_level_count(slide)

    if level_count == 4:
        lvl = 1
        ds = reader.get_downsample_ratio(wsi=slide, level=lvl)
    elif level_count == 3:
        lvl = 0
        ds = 2
    print(reader.get_mpp(slide, level=lvl), lvl)
    image, metadata = reader.get_data(wsi=slide, level=lvl)
    print(f'processing {filename}')
    print(f'metadata: {metadata}')

    if lvl == 0:
        image = cv2.resize(image.transpose(1, 2, 0), dsize=None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    H, W  = image.shape[-2:]
    print(image.shape)

    if H < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, H - tile_size + 1, stride))
        if (H - tile_size) % stride != 0:
            y_starts.append(H - tile_size)

    if W < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, W - tile_size + 1, stride))
        if (W - tile_size) % stride != 0:
            x_starts.append(W - tile_size)
    for y in y_starts:
        for x in x_starts:
            filename_ = filename.replace('.svs', ',') + f'x={math.floor(x * ds)},y={math.floor(y * ds)}.png'
            crop = image[..., y:y+tile_size, x:x+tile_size]
            
            bg = (crop[0, ...] >= threshold) & (crop[1, ...] >= threshold) & (crop[2, ...] >= threshold)
            fg = ~bg
            if np.mean(fg)<=ratio:
                continue

            mask = read_mask(geodata=geodata, location=[y*ds, x*ds], roi_size=(tile_size, tile_size), ds=ds)
            # if np.sum(mask) == 0:
            #     continue
            # CHW
            crop = Image.fromarray(crop.transpose(1, 2, 0))

            crop.save(os.path.join(export_dir, 'images', filename_))
            mask.save(os.path.join(export_dir, 'annotations', filename_))

In [ ]:

export_dir = '/mnt/d/A_cctgs_10x_s1024_patches/'
anno_prefix = ''
img_prefix = ''
os.makedirs(os.path.join(export_dir, anno_prefix, 'annotations'), exist_ok=True)
os.makedirs(os.path.join(export_dir, img_prefix, 'images'), exist_ok=True)
data = [os.path.join(svs_roots, item.replace('.png', '.svs')) for item in train_data_list]
stride = tile_size
for item in tqdm.tqdm(data):
    filename = os.path.basename(item)
    slide = reader.read(item)
    geodata = json.load(open(item.replace('.svs', '.geojson'), 'r'))

    extract_patch(reader=reader, slide=slide, geodata=geodata, filename=filename, split='')